# Лабораторна робота 5 — Логістична регресія для аналізу тональності

**Набори даних:** `amazon_baby_subset.csv`, `important_words.json`  
**Обмеження:** scikit-learn-класифікатори **не дозволені** для базових завдань.

## Налаштування

In [1]:
import sys
!{sys.executable} -m pip install numpy pandas matplotlib --quiet



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import numpy as np
import pandas as pd
import json
import matplotlib.pyplot as plt

%matplotlib inline


## Теоретичне підґрунтя

Сигмоїдна функція:
```
P(y = +1 | x, w) = 1 / (1 + exp(−wᵀ h(x)))
```
Похідна логарифму правдоподібності відносно wⱼ:
```
d(ll)/dwⱼ = dot( hⱼ,  1[y=+1] − P(y=+1|x,w) )
```
де 1[y=+1] = 1, якщо мітка +1, інакше 0.

---
## Завдання 1 — Підготовка ознак

1. Завантажте `amazon_baby_subset.csv`. Видаліть рядки з відсутніми відгуками. Вилучіть відгуки з `rating == 3`. Створіть стовпець `sentiment`: **+1** якщо `rating >= 4`, інакше **−1**.
2. Завантажте `important_words.json` (193 слова). Для кожного слова додайте стовпець до DataFrame із підрахунком його входжень у очищений текст відгуку.
3. Повідомте, скільки відгуків залишилось та який баланс класів.

In [4]:
# Завантаження набору даних
products = pd.read_csv('amazon_baby_subset.csv')

# Видаліть рядки з відсутніми відгуками та нейтральними рейтингами
products = products.dropna(subset=['review'])
products = products[products['rating'] != 3]

# Створіть стовпець sentiment: +1 якщо рейтинг >= 4, інакше -1
products['sentiment'] = products['rating'].apply(lambda x: 1 if x >= 4 else -1)

print(f'Усього відгуків : {len(products)}')
print(f'Позитивні (+1)  : {(products["sentiment"] == 1).sum()}')
print(f'Негативні (-1)  : {(products["sentiment"] == -1).sum()}')


Усього відгуків : 52831
Позитивні (+1)  : 26438
Негативні (-1)  : 26393


In [5]:
# Завантаження важливих слів
with open('important_words.json') as f:
    important_words = json.load(f)
print(f'Розмір словника: {len(important_words)} слів')


Розмір словника: 193 слів


In [6]:
# Очищення тексту відгуків (видалення пунктуації, приведення до нижнього регістру)
import string
products['review_clean'] = (
    products['review']
    .fillna('')
    .str.replace(f'[{string.punctuation}]', '', regex=True)
    .str.lower()
)

# Додайте стовпець підрахунку слів для кожного важливого слова
for word in important_words:
    products[word] = products['review_clean'].apply(
        lambda text: text.split().count(word)
    )

print('Приклад підрахунку слів:')
products[important_words[:5]].head(3)


C:\Users\test_\AppData\Local\Temp\ipykernel_12524\3319387368.py:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  products[word] = products['review_clean'].apply(
C:\Users\test_\AppData\Local\Temp\ipykernel_12524\3319387368.py:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  products[word] = products['review_clean'].apply(
C:\Users\test_\AppData\Local\Temp\ipykernel_12524\3319387368.py:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor perform

Приклад підрахунку слів:


C:\Users\test_\AppData\Local\Temp\ipykernel_12524\3319387368.py:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  products[word] = products['review_clean'].apply(


,baby,one,great,love,use
0,0,0,1,0,0
1,0,0,0,0,0
2,1,0,0,0,0


---
## Завдання 2 — Побудова матриці ознак

Реалізуйте `get_feature_matrix(df, word_list)`, яка:
1. Створює масив NumPy зі стовпцем одиниць (вільний член), за яким іде по одному стовпцю для кожного слова зі списку.
2. Повертає `(feature_matrix, sentiment_array)`, де `sentiment_array` містить +1 або −1.

Перевірка: `feature_matrix` має форму `(N, 194)`.

In [7]:
def get_feature_matrix(df, word_list):
    """
    Будує матрицю ознак та вектор міток тональності.

    Повертає
    -------
    feature_matrix  : np.ndarray, shape (n, len(word_list)+1)
    sentiment_array : np.ndarray, shape (n,)  значення {+1, -1}
    """
    # Intercept column of 1s
    df['constant'] = 1.0

    # Word feature columns
    features = ['constant'] + word_list
    feature_matrix = df[features].to_numpy()

    sentiment_array = df['sentiment'].to_numpy()
    return feature_matrix, sentiment_array


In [8]:
feature_matrix, sentiment = get_feature_matrix(products, important_words)
print(f'Розмір feature_matrix: {feature_matrix.shape}')  # очікується (N, 194)


Розмір feature_matrix: (52831, 194)


C:\Users\test_\AppData\Local\Temp\ipykernel_12524\1145285271.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['constant'] = 1.0


---
## Завдання 3 — Сигмоїдна функція та передбачення

Реалізуйте `predict_probability(feature_matrix, coefficients)`, що обчислює сигмоїдну функцію для кожного рядка. Результат — масив NumPy зі значеннями у (0, 1).

In [9]:
def predict_probability(feature_matrix, coefficients):
    """
    Обчислює P(y = +1 | x, w) для кожного рядка.

    Повертає
    -------
    probabilities : np.ndarray, shape (n,), значення у (0, 1)
    """
    # Compute the linear score for each example
    score = np.dot(feature_matrix, coefficients)

    # Apply sigmoid
    probabilities = 1.0 / (1.0 + np.exp(-score))
    return probabilities


### Перевірка — при нульових вхідних даних кожне передбачення має дорівнювати 0.5

In [10]:
zero_coeffs = np.zeros(feature_matrix.shape[1])
test_probs  = predict_probability(feature_matrix, zero_coeffs)
print(f'Усі передбачення 0.5: {np.allclose(test_probs, 0.5)}')


Усі передбачення 0.5: True


---
## Завдання 4 — Градієнтний підйом

Реалізуйте `logistic_regression(feature_matrix, sentiment, initial_coefficients, step_size, max_iter)`. На кожній ітерації:
1. Обчислюйте передбачення за допомогою `predict_probability()`.
2. Обчислюйте `errors = 1[y=+1] − predictions`.
3. Для кожного коефіцієнта j: `derivative = dot(feature_j, errors)`, потім `coeff[j] += step_size · derivative`.

Запустіть з: `initial_coefficients = np.zeros(194)`, `step_size = 1e-7`, `max_iter = 301`. Виводьте логарифм правдоподібності кожні 50 ітерацій — він має зростати монотонно.

In [11]:
def compute_log_likelihood(feature_matrix, sentiment, coefficients):
    """Допоміжна функція (надана) — обчислює логарифм правдоподібності для моніторингу."""
    indicator = (sentiment == +1).astype(float)
    scores    = np.dot(feature_matrix, coefficients)
    ll        = np.sum(
        indicator * scores - np.log(1.0 + np.exp(scores))
    )
    return ll


In [12]:
def logistic_regression(feature_matrix, sentiment,
                        initial_coefficients, step_size, max_iter):
    """
    Навчає ваги логістичної регресії методом градієнтного підйому.

    Повертає
    -------
    coefficients : np.ndarray, shape (n_features,)
    """
    coefficients = np.array(initial_coefficients, dtype=float)
    indicator    = (sentiment == +1).astype(float)   # 1 if positive, 0 otherwise

    for itr in range(max_iter):
        # 1. Compute predictions P(y=+1 | x, w)
        predictions = predict_probability(feature_matrix, coefficients)

        # 2. Compute errors = indicator(y=+1) - predictions
        errors = indicator - predictions

        # 3. Update every coefficient
        for j in range(len(coefficients)):
            derivative = np.dot(feature_matrix[:, j], errors)
            coefficients[j] += step_size * derivative

        # Print log-likelihood every 50 iterations
        if itr % 50 == 0:
            ll = compute_log_likelihood(feature_matrix, sentiment, coefficients)
            print(f'Ітерація {itr:4d}  |  логарифм правдоподібності: {ll:.4f}')

    return coefficients

### Запуск моделі

In [13]:
coefficients = logistic_regression(
    feature_matrix, sentiment,
    initial_coefficients=np.zeros(194),
    step_size=1e-7,
    max_iter=301
)


Ітерація    0  |  логарифм правдоподібності: -36612.6579
Ітерація   50  |  логарифм правдоподібності: -36272.1074
Ітерація  100  |  логарифм правдоподібності: -35948.8233
Ітерація  150  |  логарифм правдоподібності: -35641.1938
Ітерація  200  |  логарифм правдоподібності: -35347.9433
Ітерація  250  |  логарифм правдоподібності: -35068.0110
Ітерація  300  |  логарифм правдоподібності: -34800.4820


### Точність класифікації та базовий рівень

In [14]:
# Передбачте мітки класів (+1 якщо score > 0, інакше -1)
scores          = np.dot(feature_matrix, coefficients)
predictions     = np.where(scores > 0, +1, -1)

# Точність моделі
model_accuracy  = np.mean(predictions == sentiment)
print(f'Точність моделі   : {model_accuracy:.4f}')

# Базовий рівень мажоритарного класу
majority_class  = +1 if (sentiment == +1).sum() >= (sentiment == -1).sum() else -1
baseline_acc    = np.mean(majority_class == sentiment)
print(f'Базовий рівень    : {baseline_acc:.4f}  (завжди передбачає {majority_class})')


Точність моделі   : 0.7690
Базовий рівень    : 0.5004  (завжди передбачає 1)


---
## ✨ Бонус — Інтерпретація моделі

Зіставте кожне слово з його навченим коефіцієнтом. Виведіть 10 слів з найбільшими коефіцієнтами та 10 слів з найменшими.

Для одного слова з кожного списку знайдіть відгук у наборі даних, що його містить, і вкажіть передбачувану ймовірність моделі.

In [ ]:
# Бонус — аналіз коефіцієнтів
# Підказка: створіть DataFrame зі стовпцями ['word', 'coefficient']
word_coef_df = pd.DataFrame({
    'word': ['constant'] + important_words,
    'coefficient': coefficients
})

word_coef_df = word_coef_df[word_coef_df['word'] != 'constant']

# 10 найбільш позитивних слів
top_positive = word_coef_df.sort_values(by='coefficient', ascending=False).head(10)
print("Найбільш позитивні слова:\n", top_positive)

print("\n" + "-"*40 + "\n")

# 10 найбільш негативних слів
top_negative = word_coef_df.sort_values(by='coefficient', ascending=True).head(10)
print("Найбільш негативні слова:\n", top_negative)

Найбільш позитивні слова:
        word  coefficient
4      love     0.084133
3     great     0.083131
8      easy     0.073516
23    loves     0.048089
9    little     0.045382
34  perfect     0.034444
12     well     0.027673
35     nice     0.020788
11      old     0.019761
91     fits     0.019136

----------------------------------------

Найбільш негативні слова:
              word  coefficient
6           would    -0.052567
19        product    -0.042690
97          money    -0.040279
78           work    -0.033173
33           even    -0.033094
106  disappointed    -0.030481
13            get    -0.029475
29           back    -0.028210
113         waste    -0.027194
114        return    -0.026804


In [ ]:
# Бонус — приклад відгуку з передбачуваною ймовірністю
top_pos_word = top_positive.iloc[0]['word']

sample_idx = products[products[top_pos_word] > 0].index[0]
sample_review = products.loc[sample_idx, 'review']

sample_feature = feature_matrix[sample_idx : sample_idx+1]
sample_prob = predict_probability(sample_feature, coefficients)[0]

print(f"Слово: '{top_pos_word}'")
print(f"Відгук: {sample_review}")
print(f"Передбачена ймовірність позитивного класу: {sample_prob:.4f}")

Слово: 'love'
Відгук: Beautiful book, I love it to record cherished times in my great granddaughters life with the beautiful pastel pink color.
Передбачена ймовірність позитивного класу: 0.5406


**Найбільш позитивні слова:** love, great, easy, loves, little, perfect, well, nice, old, fits.

**Найбільш негативні слова:** would, product, money, work, even, disappointed, get, back, waste, return. 

**Спостереження:**

З позитивного списку: Слово "love" очікувано має найбільший коефіцієнт. Воно є найсильнішим емоційним маркером задоволення клієнта (як видно з наведеного прикладу про книгу). Також показовими є слова "easy" та "fits", що вказують на зручність у використанні, що особливо важливо для дитячих товарів.

З негативного списку: Дуже логічною є присутність слів "waste", "return" та "money" (які часто комбінуються у фрази на кшталт "waste of money"). Вони чітко сигналізують про те, що товар не виправдав очікувань покупця і викликав бажання повернути витрачені кошти. Також цікаво, що слово "would" очолює список — найімовірніше, через часте використання у конструкціях на кшталт "would not recommend" або "would not buy again".